In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path("/home/regen/Works/CPE-306")
DATASET_DIR = PROJECT_DIR / "dataset"
MODEL_DIR = DATASET_DIR / "models"
DATA_DIR = DATASET_DIR / "data"

APP_DIR = next(
    (folder for folder in (PROJECT_DIR / "app", DATASET_DIR / "app", PROJECT_DIR)
     if (folder / "prediction_app.py").is_file()),
    PROJECT_DIR / "app",
)
LOCAL_URL = "http://127.0.0.1:7860"
LOG_DIR = PROJECT_DIR / ".usedcarai"

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("App directory:", APP_DIR)
print("Select the ML kernel (/home/regen/Works/CPE-306/ML/bin/python).")


Python: /home/regen/Works/CPE-306/ML/bin/python
Dataset: /home/regen/Works/CPE-306/dataset
App directory: /home/regen/Works/CPE-306/app
Select the ML kernel (/home/regen/Works/CPE-306/ML/bin/python).


In [ ]:
import importlib.util

required_packages = {
    "gradio": "gradio", "sklearn": "scikit-learn", "requests": "requests",
    "pandas": "pandas", "numpy": "numpy", "joblib": "joblib",
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing packages in this kernel: " + ", ".join(missing_packages)
        + ". Install them using %pip install " + " ".join(missing_packages)
    )

import gradio as gr
import sklearn

print("Gradio version:", gr.__version__)
print("scikit-learn version:", sklearn.__version__)
print("=== APP FILE CHECK ===")
required_files = ["utils.py", "prediction_app.py"]
missing_files = [name for name in required_files if not (APP_DIR / name).is_file()]
for name in required_files:
    print(f"{name}: {'MISSING' if name in missing_files else 'PASS'}")
if missing_files:
    raise FileNotFoundError(
        f"Copy the original application files ({', '.join(missing_files)}) "
        f"and their supporting files into {APP_DIR}, or change APP_DIR in cell 1. "
        "The dataset contains data/model files, not the application source code."
    )


APP_ENV = os.environ.copy()
APP_ENV.update({
    "DATASET_DIR": str(DATASET_DIR),
    "DATA_DIR": str(DATA_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "GRADIO_SERVER_NAME": "127.0.0.1",
    "GRADIO_SERVER_PORT": "7860",
    "GRADIO_SHARE": "False",
})


/home/regen/Works/CPE-306/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio version: 6.29.1
scikit-learn version: 1.6.1
=== APP FILE CHECK ===
utils.py: PASS
prediction_app.py: PASS


In [ ]:
import socket
import subprocess
import time
import requests
from IPython.display import display, HTML

previous_process = globals().get("gradio_process")
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_process.kill()
        previous_process.wait()

with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 7860)) == 0:
        raise RuntimeError("Port 7860 is already in use. Stop the existing server first.")

if not all((APP_DIR / name).is_file() for name in required_files):
    raise FileNotFoundError(f"Application files are missing from {APP_DIR}")
if "APP_ENV" not in globals():
    raise RuntimeError("Run the environment check cell successfully first.")

LOG_DIR.mkdir(exist_ok=True)
GRADIO_LOG = LOG_DIR / "gradio.log"
with GRADIO_LOG.open("w") as log:
    gradio_process = subprocess.Popen(
        [sys.executable, "-u", str(APP_DIR / "prediction_app.py")],
        cwd=str(APP_DIR), env=APP_ENV,
        stdout=log, stderr=subprocess.STDOUT,
    )

print("Starting Gradio locally...")
local_ready = False
with requests.Session() as session:
    session.trust_env = False 
    for _ in range(60):
        if gradio_process.poll() is not None:
            break
        try:
            response = session.get(LOCAL_URL, timeout=2)
            if response.status_code == 200:
                local_ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(1)

if not local_ready:
    if gradio_process.poll() is None:
        gradio_process.terminate()
        try:
            gradio_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            gradio_process.kill()
            gradio_process.wait()
    print(GRADIO_LOG.read_text(errors="replace")[-12000:])
    raise RuntimeError(f"Gradio did not become ready. See {GRADIO_LOG}")

print("UsedCarAI is running:", LOCAL_URL)
print("Log:", GRADIO_LOG)
display(HTML(f'<a href="{LOCAL_URL}" target="_blank">Open UsedCarAI</a>'))


Starting Gradio locally...
UsedCarAI is running: http://127.0.0.1:7860
Log: /home/regen/Works/CPE-306/.usedcarai/gradio.log


In [ ]:

ENABLE_PUBLIC_TUNNEL = False

if ENABLE_PUBLIC_TUNNEL:
    import re
    import shutil

    cloudflared = shutil.which("cloudflared")
    if cloudflared is None:
        raise FileNotFoundError("Install cloudflared before enabling a public tunnel.")
    if not globals().get("local_ready", False) or gradio_process.poll() is not None:
        raise RuntimeError("Start the local Gradio app successfully first.")

    previous_tunnel = globals().get("cloudflare_process")
    if previous_tunnel is not None and previous_tunnel.poll() is None:
        previous_tunnel.terminate()
        try:
            previous_tunnel.wait(timeout=10)
        except subprocess.TimeoutExpired:
            previous_tunnel.kill()
            previous_tunnel.wait()

    TUNNEL_LOG = LOG_DIR / "cloudflared.log"
    with TUNNEL_LOG.open("w") as log:
        cloudflare_process = subprocess.Popen(
            [cloudflared, "tunnel", "--url", LOCAL_URL, "--no-autoupdate"],
            stdout=log, stderr=subprocess.STDOUT,
        )
    deadline = time.monotonic() + 45
    cloudflare_url = None
    while time.monotonic() < deadline:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                          TUNNEL_LOG.read_text(errors="replace"))
        if match:
            cloudflare_url = match.group(0)
            break
        if cloudflare_process.poll() is not None:
            break
        time.sleep(0.5)
    if not cloudflare_url:
        if cloudflare_process.poll() is None:
            cloudflare_process.terminate()
            try:
                cloudflare_process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                cloudflare_process.kill()
                cloudflare_process.wait()
        raise RuntimeError(f"Public URL was not created. See {TUNNEL_LOG}")
    print("Public URL:", cloudflare_url)
    print("Keep the local app and tunnel processes running while using this URL.")
else:
    print("Public tunnel disabled. Local URL:", LOCAL_URL)


Public tunnel disabled. Local URL: http://127.0.0.1:7860
